# Cross-validation sweep — BanglaPoliticalStance

Runs every model in `configs/` under **one** protocol: grouped stratified 5-fold CV, augmentation applied inside each training fold, out-of-fold predictions pooled.

**Setup:** Use a GPU runtime (**Runtime → Change runtime type → T4 GPU**).

**Dataset:** Public on HF at `kishormorol/BanglaPoliticalStance`. The annotated split (198 items) is used for evaluation. The repo code handles loading.

Each model writes `experiments/cv-<name>-<timestamp>/` with `cv.json` and `predictions.csv`. Download `experiments/` at the end — `bmpb leaderboard` builds the results table from it.

In [ ]:
!nvidia-smi -L
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Clone and install

In [ ]:
import subprocess, os

# Clean up any failed previous attempt
if os.path.exists("bangla-multimodal-political-stance"):
    subprocess.run(["rm", "-rf", "bangla-multimodal-political-stance"])

# Clone (repo is public)
!git clone --depth 1 https://github.com/kishormorol/bangla-multimodal-political-stance.git

%cd bangla-multimodal-political-stance
!git log --oneline -1

In [ ]:
%cd /content/bangla-multimodal-political-stance
!pip install -q -e . 2>&1 | tail -3
!pip install -q gdown 2>&1 | tail -1
# Verify install
!python -c "import bmpb; print('bmpb installed OK')"

## 2. The data

Downloads raw CSVs and images from Google Drive. Drive rate-limits bulk downloads,
so this cell retries up to 5 times. If it still fails, the next cell creates
`corpus.csv` directly from the raw sheets already downloaded.

In [ ]:
%cd /content/bangla-multimodal-political-stance
import time, subprocess, sys

for attempt in range(1, 6):
    print(f"\n=== bmpb data attempt {attempt}/5 ===")
    result = subprocess.run([sys.executable, "-m", "bmpb.cli", "data"], capture_output=False)
    # Check if we have enough CSVs
    from pathlib import Path
    csvs = list(Path("data/raw").glob("*.csv"))
    if len(csvs) >= 20:
        print(f"\nGot {len(csvs)} CSVs — enough to proceed.")
        break
    print(f"Got {len(csvs)} CSVs so far, retrying in 30s...")
    time.sleep(30)
else:
    print(f"\nDrive download incomplete after 5 attempts. Will use fallback in next cell.")

### Fallback: build corpus.csv from HF if Drive download was incomplete

If `bmpb ingest` fails because some CSVs are missing, this cell builds
`data/processed/corpus.csv` directly from the HF annotated split.

In [ ]:
%cd /content/bangla-multimodal-political-stance
import subprocess, sys
from pathlib import Path

# Try ingest first
result = subprocess.run([sys.executable, "-m", "bmpb.cli", "ingest"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("ingest failed, using HF fallback...")
    print(result.stderr[-500:] if result.stderr else "")

corpus_path = Path("data/processed/corpus.csv")
if not corpus_path.exists():
    print("\nBuilding corpus.csv from HF annotated split...")
    from datasets import load_dataset
    import pandas as pd

    ds = load_dataset("kishormorol/BanglaPoliticalStance", split="annotated")
    df = ds.to_pandas()

    # Map HF columns to what bmpb expects
    LABEL_NAMES = {0: "govt_critique", 1: "neutral", 2: "govt_leaning"}
    corpus = pd.DataFrame({
        "item_id": df["item_id"],
        "title": df["headline"],
        "text": df["headline"],  # text = headline for this dataset
        "label": df["label"],
        "label_name": df["label"].map(LABEL_NAMES),
        "article_label_name": "",
        "image_label_name": "",
        "outlet": df["outlet"],
        "outlet_key": df["outlet"].str.lower(),
        "date": df["date"],
        "source_url": df["source_url"],
        "image_url": "",
        "image_path": "",
        "image_kind": "",
        "has_image": False,  # no images in Colab
        "annotator_1": "",
        "annotator_2": "",
        "annotator_3": "",
        "article_label": "",
        "image_label": "",
        "text_level": "headline",
    })
    # source_index for grouping (items from same outlet+date form a group)
    corpus["source_index"] = corpus["item_id"]

    corpus_path.parent.mkdir(parents=True, exist_ok=True)
    corpus.to_csv(corpus_path, index=False)
    print(f"Created corpus.csv with {len(corpus)} items from HF dataset")
else:
    print(f"corpus.csv exists ({len(open(corpus_path).readlines())-1} items)")

# Run audit
subprocess.run([sys.executable, "-m", "bmpb.cli", "audit"])

In [ ]:
# Verify corpus is ready
import pandas as pd
df = pd.read_csv("data/processed/corpus.csv")
print(f"Corpus: {len(df)} items")
print(f"Labels: {df['label_name'].value_counts().to_dict()}")
print(f"With images: {df['has_image'].sum()}")
print("Ready for CV sweep!")

## 3. The sweep

Text encoders are quick (~2 min each); vision-language models take longer (BLIP, ViLT at 384px). Run text first so you have early numbers.

**Models evaluated:**
- **Text:** majority, tfidf_logreg, BanglaBERT, mBERT, XLM-RoBERTa, BanglaELECTRA, mT5
- **Multimodal:** CLIP, ALIGN, BLIP, ViLT, FLAVA, CountVec+ViT

In [ ]:
%cd /content/bangla-multimodal-political-stance
!PYTHON=$(which python) bash scripts/run_cv.sh configs/text

In [ ]:
%cd /content/bangla-multimodal-political-stance
!PYTHON=$(which python) bash scripts/run_cv.sh configs/multimodal

## 4. Results

`leaderboard` puts the cross-validated rows in one table and keeps the originally
published numbers separate, since those came from three different protocols and
are not comparable with these.

In [ ]:
%cd /content/bangla-multimodal-political-stance
!python -m bmpb.cli leaderboard
print(open("reports/tables/leaderboard.md").read())

## 5. Take the runs home

`experiments/` and `reports/` are what the paper's tables are built from. Commit
them back, or download the archive and unpack it into the local checkout.

In [ ]:
%cd /content/bangla-multimodal-political-stance
!tar czf /content/cv-runs.tar.gz experiments reports
from google.colab import files
files.download("/content/cv-runs.tar.gz")